# Agent 2 — Notebook 04A FINAL

## Official AQA Topic Alignment and Qdrant Payload Update

Notebooks 01–04 created a reviewed assessment knowledge base and indexed **820 approved questions** in Qdrant. The source material is organised with PMT topical codes, while Agent 1 returns official AQA references such as `3.1.1`, `3.2.2`, and `3.2.6`.

This notebook adds an official AQA reference layer before Notebook 05.


## Core design decision

The original PMT fields are retained because they describe where the source PDFs came from. Official AQA fields are added as the canonical retrieval identity.

```text
PMT source metadata                     Official retrieval metadata
-------------------                     ---------------------------
2.02 Programming Concepts       →       3.2.2 Programming concepts
1.1 Representing Algorithms     →       3.1.1 Representing algorithms
```

Nothing is renamed destructively and no source identity is lost.


## What this notebook changes

### PostgreSQL

It creates:

```text
assessment_topic_official_mappings
assessment_topic_alignment_runs
```

It adds official-reference fields to:

```text
assessment_topical_topics
assessment_topical_questions
```

### Qdrant

It adds official-reference fields to the payload of each existing question point.

### What remains unchanged

```text
Question UUIDs            unchanged
Qdrant point IDs          unchanged
PMT metadata              unchanged
MiniLM vectors            unchanged
Vector dimension          384
Distance metric           cosine
Expected Qdrant points    820
```

Payload updates do not require re-embedding.


## Specification version

The existing assessment bank is aligned to **AQA GCSE Computer Science 8525, first teaching 2020, final exams 2026**, because the stored PMT/AQA assessment material belongs to that examination period.

The version label is stored explicitly so an updated 2027 mapping can later coexist without overwriting this one.


## 1. Install dependencies


In [ ]:
%pip install -q "sqlalchemy>=2.0" "psycopg[binary]>=3.1" "qdrant-client>=1.12,<2" pandas numpy python-dotenv


## 2. Configuration

Run this notebook twice.

1. Keep `EXECUTION_MODE = "preview"` and run all cells.
2. After all 35 mappings validate, change it to `EXECUTION_MODE = "apply"`, restart the kernel, and run all cells again.

The apply run is idempotent.


In [ ]:
from __future__ import annotations

import json
import os
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from qdrant_client import QdrantClient, models
from sqlalchemy import MetaData, Table, create_engine, func, inspect, select, text, update
from sqlalchemy.dialects.postgresql import insert as pg_insert
from sqlalchemy.orm import Session

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() in {"notebooks", "notebook"} else cwd
OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / ".env")

EXECUTION_MODE = "apply"  # "preview" or "apply"

DATABASE_URL = os.getenv("AGENT2_DATABASE_URL", "").strip()
if not DATABASE_URL:
    raise RuntimeError("AGENT2_DATABASE_URL is missing from Agent2/.env")

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333").strip()
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "").strip() or None
AGENT1_COLLECTION = os.getenv("QDRANT_COLLECTION", "aqa_gcse_computer_science_8525").strip()
AGENT2_COLLECTION = (
    os.getenv("AGENT2_QDRANT_COLLECTION", "").strip()
    or f"{AGENT1_COLLECTION}_questions"
)
QDRANT_TIMEOUT_SECONDS = int(os.getenv("QDRANT_TIMEOUT_SECONDS", "30"))

SPECIFICATION_CODE = "8525"
SPECIFICATION_VERSION = "first_teaching_2020_last_exams_2026"
MAPPING_VERSION = "aqa-8525-official-reference-map-v1.0.0"
MAPPING_SOURCE_URL = "https://www.aqa.org.uk/subjects/computer-science/gcse/computer-science-8525"

EXPECTED_TOPIC_COUNT = 35
EXPECTED_QDRANT_POINTS = 820
EXPECTED_VECTOR_SIZE = 384
STRICT_EXPECTED_COUNTS = True

if EXECUTION_MODE not in {"preview", "apply"}:
    raise ValueError("EXECUTION_MODE must be 'preview' or 'apply'")

print("Execution mode:", EXECUTION_MODE)
print("Agent 2 collection:", AGENT2_COLLECTION)
print("Mapping version:", MAPPING_VERSION)


## 3. Connect to PostgreSQL and local Docker Qdrant


In [ ]:
engine = create_engine(DATABASE_URL, pool_pre_ping=True, future=True)

source_metadata = MetaData()
topics = Table("assessment_topical_topics", source_metadata, autoload_with=engine)
questions = Table("assessment_topical_questions", source_metadata, autoload_with=engine)
vector_index = Table("assessment_question_vector_index", source_metadata, autoload_with=engine)

with engine.connect() as connection:
    connection.exec_driver_sql("SELECT 1")

qdrant_kwargs = {
    "url": QDRANT_URL,
    "timeout": QDRANT_TIMEOUT_SECONDS,
}
if QDRANT_API_KEY:
    qdrant_kwargs["api_key"] = QDRANT_API_KEY

qdrant_client = QdrantClient(**qdrant_kwargs)
collection_names = {item.name for item in qdrant_client.get_collections().collections}

if AGENT2_COLLECTION not in collection_names:
    raise RuntimeError(f"Qdrant collection not found: {AGENT2_COLLECTION}")

print("PostgreSQL connection successful.")
print("Qdrant connection successful.")


## 4. Canonical 35-topic mapping

The mapping below covers every PMT topical pack currently registered in PostgreSQL. Most rows map directly to an official AQA subsection. Network fundamentals (`3.5`) and ethical/legal/environmental impacts (`3.8`) are complete official sections, so their reference level is `section`.


In [ ]:
CANONICAL_MAPPINGS = [{'expected_pmt_subtopic_code': '1.1',
  'expected_pmt_subtopic_name': 'Representing Algorithms',
  'official_section_reference': '3.1',
  'official_section_name': 'Fundamentals of algorithms',
  'official_reference': '3.1.1',
  'official_concept_name': 'Representing algorithms',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '1.2',
  'expected_pmt_subtopic_name': 'Efficiency of Algorithms',
  'official_section_reference': '3.1',
  'official_section_name': 'Fundamentals of algorithms',
  'official_reference': '3.1.2',
  'official_concept_name': 'Efficiency of algorithms',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '1.3',
  'expected_pmt_subtopic_name': 'Searching Algorithms',
  'official_section_reference': '3.1',
  'official_section_name': 'Fundamentals of algorithms',
  'official_reference': '3.1.3',
  'official_concept_name': 'Searching algorithms',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '1.4',
  'expected_pmt_subtopic_name': 'Sorting Algorithms',
  'official_section_reference': '3.1',
  'official_section_name': 'Fundamentals of algorithms',
  'official_reference': '3.1.4',
  'official_concept_name': 'Sorting algorithms',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.01',
  'expected_pmt_subtopic_name': 'Data Types',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.1',
  'official_concept_name': 'Data types',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.02',
  'expected_pmt_subtopic_name': 'Programming Concepts',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.2',
  'official_concept_name': 'Programming concepts',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.03',
  'expected_pmt_subtopic_name': 'Arithmetic Operations',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.3',
  'official_concept_name': 'Arithmetic operations in a programming language',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.04',
  'expected_pmt_subtopic_name': 'Relational Operations',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.4',
  'official_concept_name': 'Relational operations in a programming language',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.05',
  'expected_pmt_subtopic_name': 'Boolean Operations',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.5',
  'official_concept_name': 'Boolean operations in a programming language',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.06',
  'expected_pmt_subtopic_name': 'Data Structures',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.6',
  'official_concept_name': 'Data structures',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.07',
  'expected_pmt_subtopic_name': 'Input or Output',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.7',
  'official_concept_name': 'Input/output',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.08',
  'expected_pmt_subtopic_name': 'String Handling',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.8',
  'official_concept_name': 'String handling operations in a programming language',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.09',
  'expected_pmt_subtopic_name': 'Random Number Generation',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.9',
  'official_concept_name': 'Random number generation in a programming language',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.10',
  'expected_pmt_subtopic_name': 'Structured Programming and Subroutines',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.10',
  'official_concept_name': 'Structured programming and subroutines (procedures and functions)',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '2.11',
  'expected_pmt_subtopic_name': 'Robust and Secure Programming',
  'official_section_reference': '3.2',
  'official_section_name': 'Programming',
  'official_reference': '3.2.11',
  'official_concept_name': 'Robust and secure programming',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '3.1',
  'expected_pmt_subtopic_name': 'Number Bases',
  'official_section_reference': '3.3',
  'official_section_name': 'Fundamentals of data representation',
  'official_reference': '3.3.1',
  'official_concept_name': 'Number bases',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '3.2',
  'expected_pmt_subtopic_name': 'Converting Between Number Bases',
  'official_section_reference': '3.3',
  'official_section_name': 'Fundamentals of data representation',
  'official_reference': '3.3.2',
  'official_concept_name': 'Converting between number bases',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '3.3',
  'expected_pmt_subtopic_name': 'Units of Information',
  'official_section_reference': '3.3',
  'official_section_name': 'Fundamentals of data representation',
  'official_reference': '3.3.3',
  'official_concept_name': 'Units of information',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '3.4',
  'expected_pmt_subtopic_name': 'Binary Arithmetic',
  'official_section_reference': '3.3',
  'official_section_name': 'Fundamentals of data representation',
  'official_reference': '3.3.4',
  'official_concept_name': 'Binary arithmetic',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '3.5',
  'expected_pmt_subtopic_name': 'Character Encoding',
  'official_section_reference': '3.3',
  'official_section_name': 'Fundamentals of data representation',
  'official_reference': '3.3.5',
  'official_concept_name': 'Character encoding',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '3.6',
  'expected_pmt_subtopic_name': 'Representing Images',
  'official_section_reference': '3.3',
  'official_section_name': 'Fundamentals of data representation',
  'official_reference': '3.3.6',
  'official_concept_name': 'Representing images',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '3.7',
  'expected_pmt_subtopic_name': 'Representing Sound',
  'official_section_reference': '3.3',
  'official_section_name': 'Fundamentals of data representation',
  'official_reference': '3.3.7',
  'official_concept_name': 'Representing sound',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '3.8',
  'expected_pmt_subtopic_name': 'Data Compression',
  'official_section_reference': '3.3',
  'official_section_name': 'Fundamentals of data representation',
  'official_reference': '3.3.8',
  'official_concept_name': 'Data compression',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '4.1',
  'expected_pmt_subtopic_name': 'Hardware and Software',
  'official_section_reference': '3.4',
  'official_section_name': 'Computer systems',
  'official_reference': '3.4.1',
  'official_concept_name': 'Hardware and software',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '4.2',
  'expected_pmt_subtopic_name': 'Boolean Logic',
  'official_section_reference': '3.4',
  'official_section_name': 'Computer systems',
  'official_reference': '3.4.2',
  'official_concept_name': 'Boolean logic',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '4.3',
  'expected_pmt_subtopic_name': 'Software Classification',
  'official_section_reference': '3.4',
  'official_section_name': 'Computer systems',
  'official_reference': '3.4.3',
  'official_concept_name': 'Software classification',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '4.4',
  'expected_pmt_subtopic_name': 'Classification of Programming Languages and Translators',
  'official_section_reference': '3.4',
  'official_section_name': 'Computer systems',
  'official_reference': '3.4.4',
  'official_concept_name': 'Classification of programming languages and translators',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '4.5',
  'expected_pmt_subtopic_name': 'Systems Architecture',
  'official_section_reference': '3.4',
  'official_section_name': 'Computer systems',
  'official_reference': '3.4.5',
  'official_concept_name': 'Systems architecture',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '5',
  'expected_pmt_subtopic_name': 'Fundamentals of Computer Networks',
  'official_section_reference': '3.5',
  'official_section_name': 'Fundamentals of computer networks',
  'official_reference': '3.5',
  'official_concept_name': 'Fundamentals of computer networks',
  'official_reference_level': 'section',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '6.1',
  'expected_pmt_subtopic_name': 'Fundamentals of Cyber Security',
  'official_section_reference': '3.6',
  'official_section_name': 'Cyber security',
  'official_reference': '3.6.1',
  'official_concept_name': 'Fundamentals of cyber security',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '6.2',
  'expected_pmt_subtopic_name': 'Cyber Security Threats',
  'official_section_reference': '3.6',
  'official_section_name': 'Cyber security',
  'official_reference': '3.6.2',
  'official_concept_name': 'Cyber security threats',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '6.3',
  'expected_pmt_subtopic_name': 'Methods to Detect and Prevent Cyber Security Threats',
  'official_section_reference': '3.6',
  'official_section_name': 'Cyber security',
  'official_reference': '3.6.3',
  'official_concept_name': 'Methods to detect and prevent cyber security threats',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '7.1',
  'expected_pmt_subtopic_name': 'Relational Databases',
  'official_section_reference': '3.7',
  'official_section_name': 'Relational databases and structured query language (SQL)',
  'official_reference': '3.7.1',
  'official_concept_name': 'Relational databases',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '7.2',
  'expected_pmt_subtopic_name': 'Structured Query Language (SQL)',
  'official_section_reference': '3.7',
  'official_section_name': 'Relational databases and structured query language (SQL)',
  'official_reference': '3.7.2',
  'official_concept_name': 'Structured query language (SQL)',
  'official_reference_level': 'subsection',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True},
 {'expected_pmt_subtopic_code': '8',
  'expected_pmt_subtopic_name': 'Ethical, Legal and Environmental Impacts of Digital Technology on Society',
  'official_section_reference': '3.8',
  'official_section_name': 'Ethical, legal and environmental impacts of digital technology on wider society, '
                           'including issues of privacy',
  'official_reference': '3.8',
  'official_concept_name': 'Ethical, legal and environmental impacts of digital technology on wider society, '
                           'including issues of privacy',
  'official_reference_level': 'section',
  'mapping_type': 'exact',
  'mapping_status': 'approved',
  'human_approved': True}]

canonical_df = pd.DataFrame(CANONICAL_MAPPINGS)
canonical_df["official_specification_code"] = SPECIFICATION_CODE
canonical_df["official_specification_version"] = SPECIFICATION_VERSION
canonical_df["mapping_version"] = MAPPING_VERSION
canonical_df["source_url"] = MAPPING_SOURCE_URL
canonical_df["notes"] = None

print("Canonical mapping rows:", len(canonical_df))
display(
    canonical_df[
        [
            "expected_pmt_subtopic_code",
            "expected_pmt_subtopic_name",
            "official_reference",
            "official_concept_name",
            "official_reference_level",
        ]
    ]
)


## 5. Load source topics and question counts

This cell reads the existing PMT topic rows and counts how many parsed records belong to each topic. Matching uses a normalised topic name; PMT code formatting is checked separately.


In [ ]:
def normalize_label(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(value or "").lower()).strip()

def normalize_code(value: Any) -> str:
    raw = str(value or "").strip()
    if not raw:
        return ""
    parts = raw.split(".")
    return ".".join(str(int(part)) if part.isdigit() else part.lower() for part in parts)

topic_query = select(
    topics.c.id.label("topic_id"),
    topics.c.pmt_topic_number,
    topics.c.pmt_topic_name,
    topics.c.pmt_subtopic_code,
    topics.c.pmt_subtopic_name,
    topics.c.paper_code,
    topics.c.programming_language,
)
if "is_active" in topics.c:
    topic_query = topic_query.where(topics.c.is_active.is_(True))

with engine.connect() as connection:
    database_topics_df = pd.read_sql(topic_query, connection)
    question_rows_df = pd.read_sql(
        select(
            questions.c.id.label("question_id"),
            questions.c.topic_id,
            questions.c.record_type,
            questions.c.review_status,
            questions.c.retrieval_enabled,
            questions.c.is_active,
            questions.c.is_legacy,
            questions.c.embedding_status,
        ),
        connection,
    )

database_topics_df["topic_id"] = database_topics_df["topic_id"].astype(str)
question_rows_df["topic_id"] = question_rows_df["topic_id"].astype(str)

database_topics_df["_name_key"] = database_topics_df["pmt_subtopic_name"].map(normalize_label)
database_topics_df["_code_key"] = database_topics_df["pmt_subtopic_code"].map(normalize_code)
canonical_df["_name_key"] = canonical_df["expected_pmt_subtopic_name"].map(normalize_label)
canonical_df["_expected_code_key"] = canonical_df["expected_pmt_subtopic_code"].map(normalize_code)

question_counts_df = (
    question_rows_df.groupby("topic_id")
    .agg(
        total_question_records=("question_id", "count"),
        indexed_records=("embedding_status", lambda values: int((values == "indexed").sum())),
    )
    .reset_index()
)

print("Database topics:", len(database_topics_df))
print("All parsed question records:", len(question_rows_df))
print("Indexed question records:", int((question_rows_df["embedding_status"] == "indexed").sum()))


## 6. Validate one-to-one mapping coverage


In [ ]:
if database_topics_df["_name_key"].duplicated().any():
    raise RuntimeError("Duplicate PMT topic names exist in PostgreSQL.")

if canonical_df["_name_key"].duplicated().any():
    raise RuntimeError("Duplicate names exist in the canonical mapping.")

mapping_preview_df = database_topics_df.merge(
    canonical_df,
    on="_name_key",
    how="outer",
    indicator=True,
    validate="one_to_one",
)

mapping_preview_df["code_matches_after_normalisation"] = (
    mapping_preview_df["_code_key"] == mapping_preview_df["_expected_code_key"]
)

unmapped_database_df = mapping_preview_df[mapping_preview_df["_merge"] == "left_only"]
missing_canonical_df = mapping_preview_df[mapping_preview_df["_merge"] == "right_only"]
matched_df = mapping_preview_df[mapping_preview_df["_merge"] == "both"].copy()

checks_df = pd.DataFrame(
    [
        {"check": "database_topic_count", "value": len(database_topics_df), "expected": 35, "passed": len(database_topics_df) == 35},
        {"check": "canonical_mapping_count", "value": len(canonical_df), "expected": 35, "passed": len(canonical_df) == 35},
        {"check": "matched_topic_count", "value": len(matched_df), "expected": 35, "passed": len(matched_df) == 35},
        {"check": "unmapped_database_topics", "value": len(unmapped_database_df), "expected": 0, "passed": unmapped_database_df.empty},
        {"check": "missing_canonical_topics", "value": len(missing_canonical_df), "expected": 0, "passed": missing_canonical_df.empty},
        {"check": "unique_official_references", "value": matched_df["official_reference"].nunique(), "expected": 35, "passed": matched_df["official_reference"].nunique() == 35},
    ]
)

display(checks_df)

if not checks_df["passed"].all():
    if not unmapped_database_df.empty:
        display(unmapped_database_df)
    if not missing_canonical_df.empty:
        display(missing_canonical_df)
    raise RuntimeError("Topic alignment validation failed. No writes were performed.")

code_warnings_df = matched_df[~matched_df["code_matches_after_normalisation"]]
if not code_warnings_df.empty:
    print("Code-format warnings found; names still matched one-to-one.")
    display(code_warnings_df[["pmt_subtopic_code", "expected_pmt_subtopic_code", "pmt_subtopic_name"]])

matched_df = matched_df.merge(question_counts_df, on="topic_id", how="left", validate="one_to_one")
matched_df[["total_question_records", "indexed_records"]] = (
    matched_df[["total_question_records", "indexed_records"]].fillna(0).astype(int)
)

final_mapping_df = matched_df[
    [
        "topic_id",
        "pmt_topic_number",
        "pmt_topic_name",
        "pmt_subtopic_code",
        "pmt_subtopic_name",
        "official_section_reference",
        "official_section_name",
        "official_reference",
        "official_concept_name",
        "official_reference_level",
        "mapping_type",
        "mapping_status",
        "human_approved",
        "mapping_version",
        "source_url",
        "notes",
        "total_question_records",
        "indexed_records",
    ]
].sort_values(["official_section_reference", "official_reference"]).reset_index(drop=True)

display(final_mapping_df)
print("All 35 topics mapped successfully.")


## 7. Human approval gate

Before apply mode, verify:

```text
mapped topics                35
unmapped topics               0
missing canonical topics      0
unique official references   35
indexed questions            820
```


In [ ]:
indexed_count = int((question_rows_df["embedding_status"] == "indexed").sum())

if STRICT_EXPECTED_COUNTS and indexed_count != EXPECTED_QDRANT_POINTS:
    raise RuntimeError(
        f"Expected {EXPECTED_QDRANT_POINTS} indexed questions, found {indexed_count}."
    )

print("Mapping approval gate passed.")
if EXECUTION_MODE == "preview":
    print('Preview mode: no PostgreSQL or Qdrant writes will occur.')


## 8. Prepare PostgreSQL schema

This runs only in apply mode. It adds official fields without removing PMT fields and creates audit/mapping tables.


### Apply-mode safety

The schema preparation is deliberately idempotent. In `apply` mode it is run:

1. in the schema section; and
2. once again immediately before table reflection and data storage.

This prevents `NoSuchTableError` when the configuration cell is changed from
`preview` to `apply` without recreating the kernel state in the expected order.


In [ ]:
TOPIC_COLUMNS = {
    "official_specification_code": "VARCHAR(20)",
    "official_specification_version": "VARCHAR(120)",
    "official_section_reference": "VARCHAR(20)",
    "official_section_name": "TEXT",
    "official_reference": "VARCHAR(20)",
    "official_concept_name": "TEXT",
    "official_reference_level": "VARCHAR(30)",
    "official_mapping_type": "VARCHAR(30)",
    "official_mapping_status": "VARCHAR(40)",
    "official_mapping_version": "VARCHAR(120)",
    "official_mapping_human_approved": "BOOLEAN",
    "official_mapped_at": "TIMESTAMPTZ",
}

QUESTION_COLUMNS = {
    "official_specification_code": "VARCHAR(20)",
    "official_specification_version": "VARCHAR(120)",
    "official_section_reference": "VARCHAR(20)",
    "official_section_name": "TEXT",
    "official_reference": "VARCHAR(20)",
    "official_concept_name": "TEXT",
    "official_reference_level": "VARCHAR(30)",
    "official_mapping_version": "VARCHAR(120)",
    "official_mapped_at": "TIMESTAMPTZ",
}

schema_prepared = False


def prepare_alignment_schema() -> None:
    """
    Create/verify every PostgreSQL table and column required by
    Notebook 04A. All statements are idempotent.
    """
    with engine.begin() as connection:
        for column_name, column_type in TOPIC_COLUMNS.items():
            connection.execute(
                text(
                    f"ALTER TABLE assessment_topical_topics "
                    f"ADD COLUMN IF NOT EXISTS {column_name} {column_type}"
                )
            )

        for column_name, column_type in QUESTION_COLUMNS.items():
            connection.execute(
                text(
                    f"ALTER TABLE assessment_topical_questions "
                    f"ADD COLUMN IF NOT EXISTS {column_name} {column_type}"
                )
            )

        connection.execute(
            text(
                """
                CREATE TABLE IF NOT EXISTS assessment_topic_alignment_runs (
                    id UUID PRIMARY KEY,
                    mapping_version VARCHAR(120) NOT NULL,
                    specification_code VARCHAR(20) NOT NULL,
                    specification_version VARCHAR(120) NOT NULL,
                    status VARCHAR(40) NOT NULL,
                    source_topic_count INTEGER NOT NULL DEFAULT 0,
                    mapped_topic_count INTEGER NOT NULL DEFAULT 0,
                    question_rows_backfilled INTEGER NOT NULL DEFAULT 0,
                    qdrant_count_before INTEGER NOT NULL DEFAULT 0,
                    qdrant_points_updated INTEGER NOT NULL DEFAULT 0,
                    qdrant_count_after INTEGER NOT NULL DEFAULT 0,
                    configuration JSONB NOT NULL DEFAULT '{}'::jsonb,
                    started_at TIMESTAMPTZ NOT NULL,
                    completed_at TIMESTAMPTZ,
                    error_message TEXT
                )
                """
            )
        )

        connection.execute(
            text(
                """
                CREATE TABLE IF NOT EXISTS assessment_topic_official_mappings (
                    id UUID PRIMARY KEY,
                    topic_id UUID NOT NULL REFERENCES assessment_topical_topics(id) ON DELETE CASCADE,
                    specification_code VARCHAR(20) NOT NULL,
                    specification_version VARCHAR(120) NOT NULL,
                    pmt_topic_number VARCHAR(30),
                    pmt_topic_name TEXT,
                    pmt_subtopic_code VARCHAR(30),
                    pmt_subtopic_name TEXT NOT NULL,
                    official_section_reference VARCHAR(20) NOT NULL,
                    official_section_name TEXT NOT NULL,
                    official_reference VARCHAR(20) NOT NULL,
                    official_concept_name TEXT NOT NULL,
                    official_reference_level VARCHAR(30) NOT NULL,
                    mapping_type VARCHAR(30) NOT NULL,
                    mapping_status VARCHAR(40) NOT NULL,
                    is_primary BOOLEAN NOT NULL DEFAULT TRUE,
                    human_approved BOOLEAN NOT NULL DEFAULT FALSE,
                    mapping_version VARCHAR(120) NOT NULL,
                    source_url TEXT,
                    notes TEXT,
                    created_at TIMESTAMPTZ NOT NULL,
                    updated_at TIMESTAMPTZ NOT NULL,
                    CONSTRAINT uq_topic_official_mapping_version
                        UNIQUE (topic_id, specification_version)
                )
                """
            )
        )

        connection.execute(
            text(
                "CREATE INDEX IF NOT EXISTS ix_topic_mapping_official_reference "
                "ON assessment_topic_official_mappings "
                "(official_reference, specification_version)"
            )
        )
        connection.execute(
            text(
                "CREATE INDEX IF NOT EXISTS ix_questions_official_reference "
                "ON assessment_topical_questions (official_reference)"
            )
        )

    required_tables = {
        "assessment_topic_alignment_runs",
        "assessment_topic_official_mappings",
    }

    existing_tables = set(
        inspect(engine).get_table_names()
    )

    missing_tables = (
        required_tables - existing_tables
    )

    if missing_tables:
        raise RuntimeError(
            "Schema preparation finished but these tables "
            f"are still missing: {sorted(missing_tables)}"
        )


if EXECUTION_MODE == "preview":
    print("Schema writes skipped in preview mode.")
else:
    prepare_alignment_schema()
    schema_prepared = True
    print("PostgreSQL alignment schema created/verified.")


## 9. Upsert mapping rows and backfill PostgreSQL

Every topic gets one authoritative mapping row. The official fields are then copied to the linked question records for efficient later filtering.


In [ ]:
def utc_now() -> datetime:
    return datetime.now(timezone.utc)

alignment_run_id = None
mapped_topic_rows = 0
backfilled_question_rows = 0

if EXECUTION_MODE == "preview":
    print("PostgreSQL data writes skipped in preview mode.")
else:
    # Self-healing safeguard:
    # create/verify the schema again before reflecting new tables.
    prepare_alignment_schema()

    required_tables = {
        "assessment_topic_alignment_runs",
        "assessment_topic_official_mappings",
    }

    existing_tables = set(
        inspect(engine).get_table_names()
    )

    missing_tables = (
        required_tables - existing_tables
    )

    if missing_tables:
        raise RuntimeError(
            "Required PostgreSQL tables are missing: "
            f"{sorted(missing_tables)}"
        )

    print(
        "Required PostgreSQL alignment tables verified."
    )

    aligned_metadata = MetaData()
    aligned_topics = Table("assessment_topical_topics", aligned_metadata, autoload_with=engine)
    aligned_questions = Table("assessment_topical_questions", aligned_metadata, autoload_with=engine)
    official_mappings = Table("assessment_topic_official_mappings", aligned_metadata, autoload_with=engine)
    alignment_runs = Table("assessment_topic_alignment_runs", aligned_metadata, autoload_with=engine)

    alignment_run_id = uuid.uuid4()
    now = utc_now()

    with Session(engine) as session:
        session.execute(
            alignment_runs.insert().values(
                id=alignment_run_id,
                mapping_version=MAPPING_VERSION,
                specification_code=SPECIFICATION_CODE,
                specification_version=SPECIFICATION_VERSION,
                status="running",
                source_topic_count=len(database_topics_df),
                configuration={
                    "collection": AGENT2_COLLECTION,
                    "preserve_pmt_fields": True,
                    "regenerate_embeddings": False,
                    "payload_only_qdrant_update": True,
                },
                started_at=now,
            )
        )
        session.commit()

    try:
        with Session(engine) as session:
            for _, row in final_mapping_df.iterrows():
                topic_uuid = uuid.UUID(str(row["topic_id"]))
                mapped_at = utc_now()

                mapping_values = {
                    "topic_id": topic_uuid,
                    "specification_code": SPECIFICATION_CODE,
                    "specification_version": SPECIFICATION_VERSION,
                    "pmt_topic_number": str(row["pmt_topic_number"]),
                    "pmt_topic_name": row["pmt_topic_name"],
                    "pmt_subtopic_code": str(row["pmt_subtopic_code"]),
                    "pmt_subtopic_name": row["pmt_subtopic_name"],
                    "official_section_reference": row["official_section_reference"],
                    "official_section_name": row["official_section_name"],
                    "official_reference": row["official_reference"],
                    "official_concept_name": row["official_concept_name"],
                    "official_reference_level": row["official_reference_level"],
                    "mapping_type": row["mapping_type"],
                    "mapping_status": "approved",
                    "is_primary": True,
                    "human_approved": True,
                    "mapping_version": MAPPING_VERSION,
                    "source_url": MAPPING_SOURCE_URL,
                    "notes": None,
                    "updated_at": mapped_at,
                }

                statement = (
                    pg_insert(official_mappings)
                    .values(id=uuid.uuid4(), created_at=mapped_at, **mapping_values)
                    .on_conflict_do_update(
                        constraint="uq_topic_official_mapping_version",
                        set_=mapping_values,
                    )
                )
                session.execute(statement)

                common_values = {
                    "official_specification_code": SPECIFICATION_CODE,
                    "official_specification_version": SPECIFICATION_VERSION,
                    "official_section_reference": row["official_section_reference"],
                    "official_section_name": row["official_section_name"],
                    "official_reference": row["official_reference"],
                    "official_concept_name": row["official_concept_name"],
                    "official_reference_level": row["official_reference_level"],
                    "official_mapping_version": MAPPING_VERSION,
                    "official_mapped_at": mapped_at,
                }

                topic_result = session.execute(
                    update(aligned_topics)
                    .where(aligned_topics.c.id == topic_uuid)
                    .values(
                        **common_values,
                        official_mapping_type=row["mapping_type"],
                        official_mapping_status="approved",
                        official_mapping_human_approved=True,
                    )
                )
                mapped_topic_rows += int(topic_result.rowcount or 0)

                question_result = session.execute(
                    update(aligned_questions)
                    .where(aligned_questions.c.topic_id == topic_uuid)
                    .values(**common_values)
                )
                backfilled_question_rows += int(question_result.rowcount or 0)

            session.commit()

        with Session(engine) as session:
            session.execute(
                update(alignment_runs)
                .where(alignment_runs.c.id == alignment_run_id)
                .values(
                    status="postgresql_completed",
                    mapped_topic_count=mapped_topic_rows,
                    question_rows_backfilled=backfilled_question_rows,
                )
            )
            session.commit()

        print("Topic rows backfilled:", mapped_topic_rows)
        print("Question rows backfilled:", backfilled_question_rows)

    except Exception as error:
        with Session(engine) as session:
            session.execute(
                update(alignment_runs)
                .where(alignment_runs.c.id == alignment_run_id)
                .values(
                    status="failed",
                    error_message=f"{type(error).__name__}: {error}",
                    completed_at=utc_now(),
                )
            )
            session.commit()
        raise


## 10. Build the Qdrant payload-update plan

Only the 820 active, reviewed, non-legacy scored questions already indexed by Notebook 04 are included. Context, rejected, and legacy-disabled records stay outside Qdrant retrieval.


In [ ]:
if EXECUTION_MODE == "preview":
    indexed_payload_df = pd.DataFrame()
    payload_plan_df = final_mapping_df[
        [
            "topic_id",
            "official_reference",
            "official_concept_name",
            "indexed_records",
        ]
    ].copy()
    payload_plan_df = payload_plan_df.rename(columns={"indexed_records": "point_count"})
else:
    indexed_query = (
        select(
            aligned_questions.c.id.label("question_id"),
            aligned_questions.c.topic_id,
            aligned_questions.c.official_specification_code,
            aligned_questions.c.official_specification_version,
            aligned_questions.c.official_section_reference,
            aligned_questions.c.official_section_name,
            aligned_questions.c.official_reference,
            aligned_questions.c.official_concept_name,
            aligned_questions.c.official_reference_level,
            aligned_questions.c.official_mapping_version,
        )
        .where(
            aligned_questions.c.record_type == "scored_item",
            aligned_questions.c.review_status.in_(["human_approved", "human_corrected"]),
            aligned_questions.c.retrieval_enabled.is_(True),
            aligned_questions.c.is_active.is_(True),
            aligned_questions.c.is_legacy.is_(False),
            aligned_questions.c.embedding_status == "indexed",
        )
    )

    with engine.connect() as connection:
        indexed_payload_df = pd.read_sql(indexed_query, connection)

    indexed_payload_df["question_id"] = indexed_payload_df["question_id"].astype(str)
    indexed_payload_df["topic_id"] = indexed_payload_df["topic_id"].astype(str)

    required_fields = [
        "official_specification_code",
        "official_specification_version",
        "official_section_reference",
        "official_section_name",
        "official_reference",
        "official_concept_name",
        "official_reference_level",
        "official_mapping_version",
    ]

    if indexed_payload_df[required_fields].isna().any(axis=1).any():
        raise RuntimeError("Some indexed questions are missing official PostgreSQL fields.")

    if STRICT_EXPECTED_COUNTS and len(indexed_payload_df) != EXPECTED_QDRANT_POINTS:
        raise RuntimeError(
            f"Expected {EXPECTED_QDRANT_POINTS} indexed rows, found {len(indexed_payload_df)}."
        )

    payload_plan_df = (
        indexed_payload_df.groupby(["topic_id", *required_fields], dropna=False)
        .agg(point_count=("question_id", "count"))
        .reset_index()
        .sort_values("official_reference")
    )

display(payload_plan_df)


## 11. Update Qdrant payloads without re-embedding

`set_payload` changes metadata only. The existing 384-dimensional MiniLM vectors and point IDs remain untouched.


In [ ]:
def collection_settings():
    info = qdrant_client.get_collection(AGENT2_COLLECTION)
    vectors = info.config.params.vectors
    if isinstance(vectors, dict):
        return None, "named_vectors"
    size = int(vectors.size)
    distance = str(vectors.distance.value if hasattr(vectors.distance, "value") else vectors.distance).lower()
    return size, distance

qdrant_count_before = int(
    qdrant_client.count(collection_name=AGENT2_COLLECTION, exact=True).count
)
vector_size_before, distance_before = collection_settings()

if STRICT_EXPECTED_COUNTS and qdrant_count_before != EXPECTED_QDRANT_POINTS:
    raise RuntimeError(
        f"Expected {EXPECTED_QDRANT_POINTS} Qdrant points, found {qdrant_count_before}."
    )
if vector_size_before != EXPECTED_VECTOR_SIZE:
    raise RuntimeError(f"Unexpected vector size: {vector_size_before}")
if "cosine" not in distance_before:
    raise RuntimeError(f"Unexpected distance metric: {distance_before}")

qdrant_points_updated = 0
qdrant_update_results = []

if EXECUTION_MODE == "preview":
    print("Qdrant writes skipped in preview mode.")
else:
    payload_indexes = {
        "official_specification_code": models.PayloadSchemaType.KEYWORD,
        "official_specification_version": models.PayloadSchemaType.KEYWORD,
        "official_section_reference": models.PayloadSchemaType.KEYWORD,
        "official_reference": models.PayloadSchemaType.KEYWORD,
        "official_concept_name": models.PayloadSchemaType.KEYWORD,
        "official_reference_level": models.PayloadSchemaType.KEYWORD,
        "official_mapping_version": models.PayloadSchemaType.KEYWORD,
    }

    for field_name, schema_type in payload_indexes.items():
        try:
            qdrant_client.create_payload_index(
                collection_name=AGENT2_COLLECTION,
                field_name=field_name,
                field_schema=schema_type,
                wait=True,
            )
        except Exception as error:
            if "already exists" not in str(error).lower():
                print(f"Payload-index warning for {field_name}: {error}")

    try:
        for _, group_row in payload_plan_df.iterrows():
            topic_id = str(group_row["topic_id"])
            group_df = indexed_payload_df[indexed_payload_df["topic_id"] == topic_id]
            point_ids = group_df["question_id"].astype(str).tolist()

            payload = {
                "official_specification_code": group_row["official_specification_code"],
                "official_specification_version": group_row["official_specification_version"],
                "official_section_reference": group_row["official_section_reference"],
                "official_section_name": group_row["official_section_name"],
                "official_reference": group_row["official_reference"],
                "official_concept_name": group_row["official_concept_name"],
                "official_reference_level": group_row["official_reference_level"],
                "official_mapping_version": group_row["official_mapping_version"],
            }

            qdrant_client.set_payload(
                collection_name=AGENT2_COLLECTION,
                payload=payload,
                points=point_ids,
                wait=True,
            )

            qdrant_points_updated += len(point_ids)
            qdrant_update_results.append(
                {
                    "official_reference": payload["official_reference"],
                    "point_count": len(point_ids),
                    "status": "updated",
                }
            )

    except Exception as error:
        with Session(engine) as session:
            session.execute(
                update(alignment_runs)
                .where(alignment_runs.c.id == alignment_run_id)
                .values(
                    status="failed",
                    qdrant_count_before=qdrant_count_before,
                    qdrant_points_updated=qdrant_points_updated,
                    error_message=f"{type(error).__name__}: {error}",
                    completed_at=utc_now(),
                )
            )
            session.commit()
        raise

display(pd.DataFrame(qdrant_update_results))
print("Qdrant points updated:", qdrant_points_updated)


## 12. Verify PostgreSQL and Qdrant

The Qdrant verification scrolls payloads only; vectors are not downloaded.


In [ ]:
postgres_verification = {
    "mapping_rows": 0,
    "mapped_topic_rows": 0,
    "mapped_question_rows": 0,
    "mapped_indexed_question_rows": 0,
}

if EXECUTION_MODE == "apply":
    with engine.connect() as connection:
        postgres_verification["mapping_rows"] = int(
            connection.scalar(
                select(func.count())
                .select_from(official_mappings)
                .where(
                    official_mappings.c.specification_version == SPECIFICATION_VERSION,
                    official_mappings.c.mapping_status == "approved",
                )
            )
            or 0
        )
        postgres_verification["mapped_topic_rows"] = int(
            connection.scalar(
                select(func.count())
                .select_from(aligned_topics)
                .where(aligned_topics.c.official_reference.is_not(None))
            )
            or 0
        )
        postgres_verification["mapped_question_rows"] = int(
            connection.scalar(
                select(func.count())
                .select_from(aligned_questions)
                .where(aligned_questions.c.official_reference.is_not(None))
            )
            or 0
        )
        postgres_verification["mapped_indexed_question_rows"] = int(
            connection.scalar(
                select(func.count())
                .select_from(aligned_questions)
                .where(
                    aligned_questions.c.official_reference.is_not(None),
                    aligned_questions.c.record_type == "scored_item",
                    aligned_questions.c.retrieval_enabled.is_(True),
                    aligned_questions.c.is_active.is_(True),
                    aligned_questions.c.is_legacy.is_(False),
                    aligned_questions.c.embedding_status == "indexed",
                )
            )
            or 0
        )

def scroll_payloads() -> pd.DataFrame:
    rows = []
    offset = None
    while True:
        points, offset = qdrant_client.scroll(
            collection_name=AGENT2_COLLECTION,
            limit=256,
            offset=offset,
            with_payload=True,
            with_vectors=False,
        )
        for point in points:
            payload = point.payload or {}
            rows.append(
                {
                    "point_id": str(point.id),
                    "question_id": payload.get("question_id"),
                    "subtopic_name": payload.get("subtopic_name"),
                    "official_reference": payload.get("official_reference"),
                    "official_concept_name": payload.get("official_concept_name"),
                    "official_specification_version": payload.get("official_specification_version"),
                    "official_mapping_version": payload.get("official_mapping_version"),
                }
            )
        if offset is None:
            break
    return pd.DataFrame(rows)

qdrant_payloads_df = scroll_payloads()
qdrant_count_after = int(
    qdrant_client.count(collection_name=AGENT2_COLLECTION, exact=True).count
)
vector_size_after, distance_after = collection_settings()

official_payload_fields = [
    "official_reference",
    "official_concept_name",
    "official_specification_version",
    "official_mapping_version",
]
missing_official_payloads = int(
    qdrant_payloads_df[official_payload_fields].isna().any(axis=1).sum()
)

display(pd.DataFrame([{"metric": key, "value": value} for key, value in postgres_verification.items()]))
display(
    pd.DataFrame(
        [
            {"metric": "qdrant_count_before", "value": qdrant_count_before},
            {"metric": "qdrant_count_after", "value": qdrant_count_after},
            {"metric": "vector_size_after", "value": vector_size_after},
            {"metric": "distance_after", "value": distance_after},
            {"metric": "missing_official_payloads", "value": missing_official_payloads},
            {"metric": "unique_official_references", "value": qdrant_payloads_df["official_reference"].nunique(dropna=True)},
        ]
    )
)
display(qdrant_payloads_df.head(20))


## 13. Final completion checks


In [ ]:
final_checks = {
    "canonical_mapping_count_is_35": len(canonical_df) == 35,
    "database_topic_count_is_35": len(database_topics_df) == 35,
    "all_database_topics_matched": len(matched_df) == 35,
    "no_unmapped_database_topics": unmapped_database_df.empty,
    "no_missing_canonical_topics": missing_canonical_df.empty,
    "qdrant_point_count_unchanged": qdrant_count_after == qdrant_count_before,
    "qdrant_has_820_points": qdrant_count_after == EXPECTED_QDRANT_POINTS,
    "vector_size_is_384": vector_size_after == EXPECTED_VECTOR_SIZE,
    "distance_is_cosine": "cosine" in distance_after,
}

if EXECUTION_MODE == "apply":
    final_checks.update(
        {
            "postgres_mapping_rows_are_35": postgres_verification["mapping_rows"] == 35,
            "postgres_topic_rows_are_35": postgres_verification["mapped_topic_rows"] == 35,
            "all_question_rows_backfilled": postgres_verification["mapped_question_rows"] == len(question_rows_df),
            "indexed_question_rows_are_820": postgres_verification["mapped_indexed_question_rows"] == 820,
            "qdrant_points_updated_are_820": qdrant_points_updated == 820,
            "qdrant_missing_official_payloads_is_0": missing_official_payloads == 0,
            "qdrant_unique_official_references_are_35": qdrant_payloads_df["official_reference"].nunique(dropna=True) == 35,
        }
    )

final_checks_df = pd.DataFrame(
    [{"check": key, "passed": bool(value)} for key, value in final_checks.items()]
)
display(final_checks_df)

notebook_04a_complete = EXECUTION_MODE == "apply" and all(final_checks.values())
print("Notebook 04A complete:", notebook_04a_complete)

if EXECUTION_MODE == "preview":
    print(
        '\nPreview passed. Change only:\n'
        'EXECUTION_MODE = "apply"\n'
        'Then restart the kernel and run all cells.'
    )


## 14. Finalise the audit run and export reports


In [ ]:
if EXECUTION_MODE == "apply" and alignment_run_id is not None:
    with Session(engine) as session:
        session.execute(
            update(alignment_runs)
            .where(alignment_runs.c.id == alignment_run_id)
            .values(
                status="completed" if notebook_04a_complete else "completed_with_failed_checks",
                qdrant_count_before=qdrant_count_before,
                qdrant_points_updated=qdrant_points_updated,
                qdrant_count_after=qdrant_count_after,
                completed_at=utc_now(),
            )
        )
        session.commit()

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
mapping_path = OUTPUT_DIR / f"agent2_official_topic_mapping_{EXECUTION_MODE}_{timestamp}.csv"
qdrant_path = OUTPUT_DIR / f"agent2_official_qdrant_verification_{EXECUTION_MODE}_{timestamp}.csv"
summary_path = OUTPUT_DIR / f"agent2_official_alignment_summary_{EXECUTION_MODE}_{timestamp}.json"

final_mapping_df.to_csv(mapping_path, index=False)
qdrant_payloads_df.to_csv(qdrant_path, index=False)

summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "execution_mode": EXECUTION_MODE,
    "specification_code": SPECIFICATION_CODE,
    "specification_version": SPECIFICATION_VERSION,
    "mapping_version": MAPPING_VERSION,
    "mapped_topic_count": len(matched_df),
    "all_postgresql_question_records": len(question_rows_df),
    "qdrant_collection": AGENT2_COLLECTION,
    "qdrant_count_before": qdrant_count_before,
    "qdrant_points_updated": qdrant_points_updated,
    "qdrant_count_after": qdrant_count_after,
    "vector_size": vector_size_after,
    "distance": distance_after,
    "missing_official_payloads": missing_official_payloads,
    "postgres_verification": postgres_verification,
    "final_checks": final_checks,
    "notebook_04a_complete": notebook_04a_complete,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved:")
print(mapping_path)
print(qdrant_path)
print(summary_path)


# Completion criteria

A successful apply run should show:

```text
Mapped topics                              35
Unmapped topics                             0
Official mapping rows                      35
All PostgreSQL question rows backfilled    yes
Mapped indexed questions                  820
Qdrant points updated                     820
Qdrant points after update                820
Missing official payloads                   0
Vector size                               384
Distance                                  cosine
Notebook 04A complete                     True
```

## How Notebook 05 will use this

Agent 1 output:

```text
Topic: Iteration
Official reference: 3.2.2
Role: supporting
```

Notebook 05 can now apply:

```text
Qdrant payload filter:
official_reference = "3.2.2"
```

It can then use MiniLM only to rank questions inside the already-correct official topic pool according to the lesson-specific concept, such as iteration.
